**TODO:明天开始从头写，一点点替代PyTorch**

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST

train_data = MNIST(root='./data', train=True, transform=transforms.ToTensor(), download=True)
train_loader = DataLoader(batch_size=64, shuffle=True, num_workers=0, dataset=train_data)

In [ ]:
import numpy as np

class ReLU:
    def __init__(self):
        self.x = None
        self.dA_dL = None

    def forward(self, x):
        self.x = x
        return np.maximum(0, x)

    def backward(self, dZ_dW):
        return dZ_dW * (self.x > 0)

class Linear:
    def __init__(self, in_dim, out_dim, lr=0.001):
        self.W = np.random.randn(in_dim, out_dim) * lr
        self.b = np.zeros(out_dim)
        self.lr = lr
        self.A_prev = None
        self.dW = None
        self.db = None

    def forward(self, A_prev):
        self.A_prev = A_prev
        return self.A_prev @ self.W + self.b

    def backward(self, dL_dZ):
        self.dW = dL_dZ.T @ self.A_prev
        self.db = np.sum(dL_dZ, axis=0)
        return dL_dZ @ self.W

    def step(self):
        self.W -= self.lr * self.dW
        self.b -= self.lr * self.db

class CrossEntropyLoss:
    def __init__(self):
        self.A = None
        self.t = None
        self.N = None

    def forward(self, A, t):
        self.A = A
        self.t = t
        self.N = A.shape[0]
        A_shifted = A - np.max(A, axis=1, keepdims=True)
        exp_A = np.exp(A_shifted)
        softmax = exp_A / np.sum(exp_A, axis=1, keepdims=True)
        loss = -np.sum(np.log(softmax[np.arange(self.N), t])) / self.N
        return loss

    def backward(self):
        A_shifted = self.A - np.max(self.A, axis=1, keepdims=True)
        exp_A = np.exp(A_shifted)
        softmax = exp_A / np.sum(exp_A, axis=1, keepdims=True)
        softmax[np.arange(self.N), self.t] -= 1
        return softmax / self.N  # 形状 (N, d_out)，这就是 dL/dA
